## Exploration of the data for the sci-kit learn conference
-   This is a binary classification task
-   Evaluated on classification accuracy
-   The training set has 1000 samples and the testing set has 9000 samples 
-   Dataset has 40 features, representing objects from two classes (labeled as 0 or 1)
-   test.csv, train.csv and train_label.csv

In [ ]:
# import os
# import os

# new_dir = 'D:\Kaggle\data-science-london-scikit-learn'
# os.chdir(new_dir)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler  
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score



In [ ]:
train_df = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
label_df = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header = None)
df_test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header = None)

In [ ]:
df_train = pd.merge(train_df, label_df, left_index=True, right_index=True)

In [ ]:
df_train.head()

### Checking dimensions of data

In [ ]:
print(f"Training data shape is: {df_train.shape}")
print(f"Testing data shape is: {df_test.shape}")

In [ ]:
df_train.describe()

### Checking any missing values

In [ ]:
df_train.isnull().sum()

### Visualizing the label of the training data

In [ ]:
plt.figure(figsize=(10, 10))
#plot a bar plot of each class 0 and 1
label_df.value_counts().plot(kind='bar')
plt.title('Class distribution of the target')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
#plot a pie chart for the same
plt.figure(figsize=(10, 10))
label_df.value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Class distribution of the target')
plt.show()

### We can say that the dataset is balanced

### Checking the Distribution of the Data

In [ ]:
#plot distribution of the features
num_features = 40
num_rows = 8
num_cols = 5

fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 4*num_rows))

for i, ax in enumerate(axs.flatten()):
    if i < num_features:
        ax.hist(df_train[df_train.columns[i + 1]], bins=20, alpha=0.7)
        ax.set_title(f'Feature {i + 1} distribution')
        ax.set_xlabel(f'Feature {i + 1}')
        ax.set_ylabel('Count')
    else:
        ax.axis('off')  # turn off extra subplots if there are more than 40 features

plt.tight_layout()
plt.show()


In [ ]:
def calculate_outlier_percentage(dataframe):
    outlier_percentage = {}
    
    for i, feature in enumerate(dataframe.columns):
        # Calculate the interquartile range (IQR)
        Q1 = dataframe[feature].quantile(0.25)
        Q3 = dataframe[feature].quantile(0.75)
        IQR = Q3 - Q1
        
        # Calculate the lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count the number of outliers
        num_outliers = len(dataframe[(dataframe[feature] < lower_bound) | (dataframe[feature] > upper_bound)])
        
        # Calculate the percentage of outliers
        percentage = (num_outliers / len(dataframe)) * 100
        
        # Store the percentage of outliers for the feature
        outlier_percentage[f'feature {i}'] = percentage
    
    return outlier_percentage

unscaled_outlier_percentage = calculate_outlier_percentage(df_train)

In [ ]:
import matplotlib.pyplot as plt

# Convert the outlier_percentage dictionary to a pandas DataFrame
df_outliers = pd.DataFrame.from_dict(unscaled_outlier_percentage, orient='index', columns=['Percentage'])

# Sort the DataFrame by the outlier percentage in descending order

# Create the horizontal bar plot
plt.figure(figsize=(10, 8))
plt.barh(df_outliers.index, df_outliers['Percentage'])
plt.title('Outlier Percentage for Each Feature')
plt.xlabel('Percentage')
plt.ylabel('Feature')
plt.show()


In [ ]:
train_df.columns

In [ ]:
df_train.columns = [i for i in range(0,40)]+ ['target']

In [ ]:
features_with_outliers = np.arange(0, 40)
scaler = RobustScaler()

# Scale the selected features
df_train_scaled = df_train.copy()
df_train_scaled[features_with_outliers] = scaler.fit_transform(df_train[features_with_outliers])



In [ ]:
#Now find the outliers percentage after scaling it and plot it 
scaled_outlier_percentage = calculate_outlier_percentage(df_train_scaled)

df_outliers_scaled = pd.DataFrame.from_dict(scaled_outlier_percentage, orient='index', columns = ['Percentage'])

#Create the horizontal bar plot 
plt.figure(figsize=(10, 8))
plt.barh(df_outliers_scaled.index, df_outliers_scaled['Percentage'])
plt.title('Outlier Percentage for Each Feature After scaling')
plt.xlabel('Percentage')
plt.ylabel('Feature')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_train_scaled.iloc[:, :-1], df_train_scaled['target'], test_size=0.2, random_state=42)

## Model Selection for the selected data

In [ ]:
training_score = []
testing_score = []

In [ ]:
def model_prediction(model):
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    training_score.append(train_accuracy)
    testing_score.append(test_accuracy)
    print(f"Training accuracy: {train_accuracy}")
    print(f"Testing accuracy: {test_accuracy}")
    

### Support Vector Classifier Model

In [ ]:
model_prediction(SVC())

### Random-Forest-Classifier Model

In [ ]:
model_prediction(RandomForestClassifier())

### Ada-Boost Classifier Model

In [ ]:
model_prediction(AdaBoostClassifier())

### Gradient-Boosting-Classifier Model

In [ ]:
model_prediction(GradientBoostingClassifier())

### LGBM Classifier model

In [ ]:
model_prediction(LGBMClassifier())

### XGBClassifier Model

In [ ]:
model_prediction(XGBClassifier())

### Cat-Boost-Classifier

In [ ]:
model_prediction(CatBoostClassifier(verbose=False))

### All Model Performance Comparison

In [ ]:
models = ["SVC", "RandomForest", "AdaBoost", "GradientBoost", "LGBM", "XGB", "CatBoost"]

In [ ]:
df = pd.DataFrame({'Model': models, 'Training Accuracy': training_score, 'Testing Accuracy': testing_score})

In [ ]:
##Plotting the above results as column-bar chart
df.plot(x='Model', y=['Training Accuracy', 'Testing Accuracy'], kind='bar', figsize=(10, 8))
plt.title('Training and Testing Accuracy for Each Model')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.show()

### Hyperparameter tuning of LGBM Model

In [ ]:
model1 = LGBMClassifier()

In [ ]:
parameters1 = {"n_estimators": [100, 200, 300, 400, 500],
               "learning_rate" : [0.01, 0.05, 0.1, 0.5, 1],
               "random_state" : [42],
               "num_leaves" : [16, 17,18] }

In [ ]:
grid_search1 = GridSearchCV(model1, parameters1, scoring='accuracy', n_jobs=-1, cv=5)

In [ ]:
grid_search1.fit(X_train, y_train)

In [ ]:
grid_search1.best_score_

In [ ]:
best_parameters1 = grid_search1.best_params_
print(best_parameters1)

### Create that LGBM Model using the best parameters

In [ ]:
model1 = LGBMClassifier(**best_parameters1)

In [ ]:
model1.fit(X_train, y_train)

In [ ]:
X_test_pred1 = model1.predict(X_test)

In [ ]:
accuracy_score(y_test, X_test_pred1)

### Hyper-Parameter Tuning of CatBoost Model

In [ ]:
model2 = CatBoostClassifier(verbose = False)

In [ ]:
parameters2 = {"learning_rate": np.arange(0.1, 0.7), 
               "random_state": [42], 
               "depth": [8,9,10], 
               "iterations": [35, 40, 50]}
grid_search2 = GridSearchCV(model2, parameters2, cv =5, n_jobs = -1, scoring='accuracy')

In [ ]:
grid_search2.fit(X_train, y_train)

In [ ]:
grid_search2.best_score_

In [ ]:
best_parameters2 = grid_search2.best_params_
best_parameters2

### Creating CatBoost Model using best params

In [ ]:
model2 = CatBoostClassifier(**best_parameters2, verbose = False)

In [ ]:
model2.fit(X_train, y_train)

In [ ]:
X_test_pred2 = model2.predict(X_test)

In [ ]:
accuracy_score(y_test, X_test_pred2)

### Hyperparameter tuning of XGBoost Model

In [ ]:
model3 = XGBClassifier()

In [ ]:
parameters3 = {"n_estimators": [50, 100,150], 
               "random_state": [42],
               "learning_rate": [0.1, 0.3, 0.5, 1.0]}

In [ ]:
grid_search3 = GridSearchCV(model3, parameters3, cv  = 5, n_jobs = -1, scoring='accuracy')

In [ ]:
grid_search3.fit(X_train, y_train)

In [ ]:
grid_search3.best_score_

In [ ]:
best_parameters3 = grid_search3.best_params_

### Creating XGBoost Model using best params

In [ ]:
model3 = XGBClassifier(**best_parameters3)

In [ ]:
model3.fit(X_train, y_train)

In [ ]:
X_test_pred3 = model3.predict(X_test)

In [ ]:
accuracy_score(y_test, X_test_pred3)

###  HyperParameter tuning of RandomForestClassifier

In [ ]:
model4 = RandomForestClassifier()

In [ ]:
parameters4 = {'n_estimators': [100, 300, 500, 550],
               'min_samples_split': [7,8,9], 
                'max_depth': [10,11,12], 
                'min_samples_leaf': [4,5,6]}
grid_search4 = GridSearchCV(model4, parameters4, cv = 5, n_jobs = -1, scoring='accuracy')

In [ ]:
grid_search4.fit(X_train, y_train)

In [ ]:
grid_search4.best_score_

In [ ]:
best_parameters4 = grid_search4.best_params_
best_parameters4

### Creating Random Forest Model using best params

In [ ]:
model4 = RandomForestClassifier(**best_parameters4)

In [ ]:
model4.fit(X_train, y_train)

In [ ]:
X_test_pred4 = model4.predict(X_test)

In [ ]:
accuracy_score(y_test, X_test_pred4)

### Stacking Classifier Model

In [ ]:
stacking_model = StackingClassifier(estimators = [('LGBM', model1),
('CatBoost', model2),
('XGBoost', model3), 
('RandomForest', model4)])

In [ ]:
stacking_model.fit(X_train, y_train)

In [ ]:
X_train_pred5 = stacking_model.predict(X_train)

In [ ]:
X_test_pred5 = stacking_model.predict(X_test)

In [ ]:
print(f'Stacking model training data is {accuracy_score(y_train, X_train_pred5)}')

In [ ]:
print(f'Stacking model testing data is {accuracy_score(y_test, X_test_pred5)}')

In [ ]:
# Create a DataFrame for the predictions
#there should be id and the preds as Solution
df_preds = pd.DataFrame({'Id': range(1, len(df_test) + 1), 'Solution': stacking_model.predict(df_test)})

In [ ]:
df_preds.to_csv('/kaggle/working/submission.csv', index=False)